# Day 3-01｜Detection IoU 與跨影格關聯
> Python 籃球運動資料分析課程  
> 本單元使用 YOLO detector 在相鄰影片 frame 的 player boxes，計算 IoU matrix 並說明 tracking association 的基本問題。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 使用實際 detector 輸出建立相鄰 frame 的 player boxes。
- 計算 IoU matrix。
- 觀察單純 IoU association 在遮擋、快速移動與漏偵時的限制。


## 課程流程
1. 讀取參考影片中的相鄰 frame。
2. 對兩個 frame 執行 YOLO detection。
3. 計算 player box 的 IoU matrix。


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

bootstrap_candidates = [
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
]
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，確認課程已同步到 /content/drive/MyDrive/basketball_hackathon/course。"
    )

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT, mount_drive=False)


In [ ]:
import numpy as np
import pandas as pd

from src.yolo_utils import (
    PLAYER_CLASS_NAMES,
    detector_model_path,
    first_reference_video,
    read_video_frame,
    run_detector_on_image,
)

VIDEO_PATH = first_reference_video(COURSE_ROOT)
MODEL_PATH = detector_model_path(COURSE_ROOT)
FRAME_A = 15
FRAME_B = 16

frame_a = read_video_frame(VIDEO_PATH, FRAME_A)
frame_b = read_video_frame(VIDEO_PATH, FRAME_B)
dets_a, _ = run_detector_on_image(MODEL_PATH, frame_a, conf=0.25, imgsz=960, frame_index=FRAME_A)
dets_b, _ = run_detector_on_image(MODEL_PATH, frame_b, conf=0.25, imgsz=960, frame_index=FRAME_B)
players_a = [d for d in dets_a if d.class_name in PLAYER_CLASS_NAMES]
players_b = [d for d in dets_b if d.class_name in PLAYER_CLASS_NAMES]

print("frame A players:", len(players_a))
print("frame B players:", len(players_b))


In [ ]:
def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union else 0.0

matrix = np.zeros((len(players_a), len(players_b)), dtype=float)
for i, det_a in enumerate(players_a):
    for j, det_b in enumerate(players_b):
        matrix[i, j] = iou(det_a.bbox_xyxy, det_b.bbox_xyxy)

pd.DataFrame(
    np.round(matrix, 3),
    index=[f"A{i}" for i in range(len(players_a))],
    columns=[f"B{j}" for j in range(len(players_b))],
)


In [ ]:
assignments = []
used_b = set()
for i in range(matrix.shape[0]):
    if matrix.shape[1] == 0:
        break
    j = int(matrix[i].argmax())
    score = float(matrix[i, j])
    if score >= 0.3 and j not in used_b:
        used_b.add(j)
        assignments.append({"frame_a_box": i, "frame_b_box": j, "iou": score})

pd.DataFrame(assignments)


ByteTrack 會同時考慮高低 confidence detections 與 track state，不只用單一 pairwise IoU。下一單元會直接在影片上產生 track ID。